<a href="https://colab.research.google.com/github/Kaveesha-Vihanga/DS_Project/blob/component-3/Component_3_implementation_roberta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
pip install pandas transformers torch tqdm

In [23]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm

# Enable tqdm for pandas to see progress
tqdm.pandas()

# 1. Load your dataset (assuming it's a tab-separated file)
df = pd.read_csv('/content/drive/MyDrive/DSGP component 3/sri_lanka_hotel_reviews_english.csv', sep=',')

# 2. Load a pre-trained sentiment analysis model
print("Loading sentiment analysis model...")
classifier = pipeline(
    "sentiment-analysis",
    model="siebert/sentiment-roberta-large-english",
    device= -1
)
print("Model loaded.")

Loading sentiment analysis model...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: siebert/sentiment-roberta-large-english
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


In [24]:
# 3. Define a function to get sentiment score
def get_sentiment_score(review_text):
    """Takes review text, returns a dict with label and score."""
    if not isinstance(review_text, str) or len(review_text.strip()) == 0:
        return {'label': 'NEUTRAL', 'score': 0.5} # Handle empty reviews

    try:
        # The model returns a list, e.g., [{'label': 'POSITIVE', 'score': 0.99}]
        result = classifier(review_text, truncation=True)[0] # truncation=True handles long reviews

        # Convert score to a standard 0-1 scale (it already is)
        # Convert label to a standard format
        return {
            'label': result['label'].upper(),
            'score': result['score']
        }
    except Exception as e:
        print(f"Error processing review: {review_text[:100]}... Error: {e}")
        return {'label': 'ERROR', 'score': None}

# 4. Apply the function to your DataFrame
# This creates a new column with the dictionary results
print("Analyzing sentiments for all reviews...")
df['sentiment_result'] = df['review_clean'].progress_apply(get_sentiment_score)

# 5. Split the dictionary into separate columns for easier analysis
df['sentiment_label'] = df['sentiment_result'].apply(lambda x: x['label'])
df['sentiment_score'] = df['sentiment_result'].apply(lambda x: x['score'])

# 6. Display the first few rows to verify the results
print(df[['review_clean', 'sentiment_label', 'sentiment_score']].head())

Analyzing sentiments for all reviews...


100%|██████████| 16252/16252 [3:01:11<00:00,  1.49it/s]

                                        review_clean sentiment_label  \
0  Good Rooms were good I was expected to check o...        NEGATIVE   
1  Good Clean and close to beach Road noise at night        POSITIVE   
2             Pls talk to politle Expensive Location        NEGATIVE   
3  AAA+++ Very friendly staff, very clean rooms a...        POSITIVE   
4  Calm, Nice place in the marine drive Really ni...        POSITIVE   

   sentiment_score  
0         0.999507  
1         0.998820  
2         0.997543  
3         0.998839  
4         0.998926  
